# Input File Requirements

Input file should be an Excel. The table consists of three parts: Symbol column, data columns, and label column.

## 1. Column Structure

###  **Symbol Column**
- **Column Name**: `Symbol`
- **Content**: Gene identifiers.
- **Position**: First column.

###  **Data Columns**
- **Column Naming Rule**: `time_dup`
  - `time`: The time point of the sample.
  - `dup`: The replication number of the sample.
- **Sorting Rules**:
  - Columns with the same `dup` value are grouped together.
  - Within each group, columns are sorted by `time` in ascending order.
  - Groups are sorted by `dup` in ascending order.
- **Example Column Names**: `0_0`, `4_0`, `8_0`, `0_1`, `4_1`, `8_1`, etc.
- **Content**: Expression values of the corresponding gene at specific times and replications.

###  **Label Column**
- **Column Name**: `label`
- **Content**: A label indicating whether the gene oscillates, with values `0` or `1`.
  - `0`: The gene does not oscillate.
  - `1`: The gene oscillates.
- **Position**: Last column.
- **Note**: If the `label` is uncertain, it can be set to `1` for all entries.

## 2. Example Table Structure

| Symbol | 0_0  | 4_0  | 8_0  | ... | label |
|--------|------|------|------|-----|-------|
| GENE1  | 0.36314 | 0.838363 | 0.850397 | ... | 0     |
| GENE2  | 3.49872 | 3.780274 | 3.770533 | ... | 0     |
| ...    | ...  | ...  | ...  | ... | ...   |

## 3. Data Filling Instructions
- **Symbol Column**: Fill in the unique identifiers for the genes.
- **Data Columns**: Fill in the expression values of the corresponding gene at specific times and replications.
- **Label Column**: Fill in `0` or `1` based on whether the gene oscillates. If uncertain, fill in `1` for all entries.

In [1]:
from utils.ts import singal_convert_to_ts

singal_convert_to_ts(
    input_csv='../example_data/example_data_t1.csv',
    output_ts='../example_data/t1/test.ts',
    problem_name='example',
    label_col='label'
)

TS file generated: ../example_data/t1/test.ts


In [2]:
import argparse
from argparse import Namespace
import random
import numpy as np
import os 
import torch
from data_provider.classfication_datasets import MultipleDataset  
from torch.utils.data import DataLoader 
from models.circaFM import CIRCAFM  
from peft import PeftModel  
from peft import LoraConfig, get_peft_model
from tqdm import tqdm 
from utils.metrics import Metric  
from tabulate import tabulate  
from datetime import datetime

def control_randomness(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = False  
    torch.backends.cudnn.benchmark = True       

d:\Software\Anaconda3\envs\circallm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_file_paths(folder_path: str) -> list:
    print(f"Current working directory: {os.getcwd()}")
    all_ts = [
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.endswith('.ts') and os.path.isfile(os.path.join(folder_path, f))
    ]
    parent_dirs = sorted(list({os.path.dirname(p) for p in all_ts}))
    file_paths = [os.path.join(d, "") for d in parent_dirs]
    print(f"Found {len(file_paths)} dataset directories containing .ts files")
    return file_paths

class Circadian_Validator:
    def __init__(self, args: Namespace):
        self.args = args
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self._init_model_config()
        self._init_model()
        self._init_dataset()
        self._init_loss_function()

    def _init_model_config(self):
        self.config_dict = {
            "task_name": "classification",
            "model_name": "CIRCAFM",
            "transformer_type": "encoder_only",
            "freeze_embedder": False,
            "freeze_encoder": False,
            "freeze_head": False,
            "learning_rate": 1e-6,
            "num_epochs": 20,
            "n_channels": 1,
            "num_class": 2,
            'reduction': 'mean',
            "d_model": None,
            "seq_len": 72,
            'enable_gradient_checkpointing': False,
            "enable_FAN": True,
            "enable_FAN_gate": True,
            "patch_len": 6,
            "patch_stride_len": 6,
            "device": "cpu",
            "transformer_backbone": "google/flan-t5-small",
            "model_kwargs": {},
            "t5_config": {
                "architectures": ["T5ForConditionalGeneration"],
                "d_ff": 1024,
                "d_kv": 64,
                "d_model": 512,
                "decoder_start_token_id": 0,
                "dropout_rate": 0.1,
                "eos_token_id": 1,
                "feed_forward_proj": "gated-gelu",
                "initializer_factor": 1.0,
                "is_encoder_decoder": True,
                "layer_norm_epsilon": 1e-06,
                "model_type": "t5",
                "n_positions": 72,
                "num_decoder_layers": 6,
                "num_heads": 8,
                "num_layers": 6,
                "output_past": True,
                "pad_token_id": 0,
                "relative_attention_max_distance": 128,
                "relative_attention_num_buckets": 32,
                "tie_word_embeddings": False,
                "use_cache": True,
                "vocab_size": 32128
            }
        }
        self.config = Namespace(**self.config_dict)
        print("Model configuration initialized")

    def _verify_loaded_weights(self):
        param_values = []
        for param in self.model.parameters():
            if param.requires_grad:
                param_values.append(param.detach().cpu().numpy().flatten())
        if not param_values:
            print("Model has no trainable parameters, possible structural error!")
            return

        all_params = np.concatenate(param_values)
        mean_val = np.mean(all_params)
        std_val = np.std(all_params)
        print(f"Trainable param stats: mean={mean_val:.6f}, std={std_val:.6f}")

        if abs(mean_val) < 1e-4:
            for i, param in enumerate(self.model.parameters()):
                if param.requires_grad:
                    print(f"Param {i} first 5 values: {param.detach().cpu().numpy().flatten()[:5]}")
                    if i >= 2:
                        break
        else:
            print("Weights loaded successfully! Trainable parameters far from initial values")

    def _init_model(self):
        self.model = CIRCAFM(self.config)
        self.model.to(self.device)
        print("Base CIRCAFM model initialized")

        if not os.path.isfile(self.args.pretrained_model_path):
            raise FileNotFoundError(f"Model file not found: {self.args.pretrained_model_path}")

        checkpoint = torch.load(
            self.args.pretrained_model_path,
            map_location=self.device,
            weights_only=True
        )
        peft_model_state = checkpoint['model_state_dict']
        print(f"Read PeftModel parameters, total keys: {len(peft_model_state)}")

        sample_key = next(iter(peft_model_state.keys()))
        if "base_model.model." not in sample_key:
            print("Warning: Loaded parameters are not in PeftModel format, may not be a LoRA model")
        else:
            print(f"PeftModel format verified, sample key: {sample_key[:50]}...")

        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q", "v"],
            lora_dropout=0.1,
            bias="none",
            task_type="SEQ_CLS"
        )
        self.model = get_peft_model(self.model, lora_config)
        self.model.to(self.device)
        print("Added LoRA structure to base model, converted to PeftModel")

        try:
            load_result = self.model.load_state_dict(peft_model_state, strict=True)
            print("Pretrained weights loaded strictly successfully!")
            if load_result.missing_keys:
                print(f"  Missing keys: {load_result.missing_keys[:3]}...")
            else:
                print("  No missing keys")
            if load_result.unexpected_keys:
                print(f"  Unexpected keys: {load_result.unexpected_keys[:3]}...")
            else:
                print("  No unexpected keys")
        except RuntimeError as e:
            print(f"Strict loading failed, attempting non-strict loading: {str(e)[:300]}")
            load_result = self.model.load_state_dict(peft_model_state, strict=False)
            print(f"  Missing keys count: {len(load_result.missing_keys)}")
            print(f"  Unexpected keys count: {len(load_result.unexpected_keys)}")
            lora_loaded = any("lora_" in key for key in peft_model_state.keys() if key not in load_result.missing_keys)
            if not lora_loaded:
                raise RuntimeError("LoRA adapter parameters not loaded successfully, validation results will be invalid!")

        self._verify_loaded_weights()
        self.model.eval()
        print("Model loaded and set to evaluation mode")

    def _init_dataset(self):
        print(f"Loading validation dataset: {self.args.dataset_path}")
        file_paths = get_file_paths(self.args.dataset_path)
        print(file_paths)

        self.val_dataset = MultipleDataset(
            data_split="aper",
            file_paths=file_paths,
            seq_len=self.args.seq_len,
            seed=self.args.seed,
            Realcase=True
        )

        self.val_dataloader = DataLoader(
            self.val_dataset,
            batch_size=self.args.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
            drop_last=False
        )
        print(f"Validation dataset loaded: {len(self.val_dataset)} samples, {len(self.val_dataloader)} batches")

    def _init_loss_function(self):
        self.criterion = torch.nn.CrossEntropyLoss()
        print("Loss function initialized")

    def evaluate(self) -> dict:
        print("Starting evaluation...")
        dataloader = self.val_dataloader

        self.model.eval()
        self.model.to(self.device)

        all_sample_ids = []
        all_targets = []
        all_preds = []
        all_scores = []
        running_loss, correct, total = 0.0, 0, 0

        with torch.no_grad():
            for batch_data, input_mask, x_marks, batch_labels in tqdm(dataloader, total=len(dataloader)):
                if isinstance(batch_labels, torch.Tensor):
                    batch_labels = batch_labels.detach().cpu().tolist()

                batch_sample_ids = [str(item) for item in batch_labels[0]]
                batch_targets_list = [int(item) for item in batch_labels[1]]

                batch_targets = torch.tensor(batch_targets_list, dtype=torch.long, device=self.device)

                all_sample_ids.extend(batch_sample_ids)
                all_targets.extend(batch_targets_list)

                batch_data = batch_data.to(self.device).float()
                input_mask, x_marks = input_mask.long().to(self.device), x_marks.to(self.device)
                total += batch_targets.size(0)

                with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float32):
                    output = self.model(x_enc=batch_data, input_mask=input_mask, x_mark=x_marks, reduction=self.args.reduction, return_dict=True)
                    logits = output.logits
                    loss = self.criterion(logits, batch_targets)
                running_loss += loss.item()

                scores = torch.softmax(logits, dim=1)
                _, predicted = torch.max(scores, 1)

                scores = scores.to(dtype=torch.float32)
                predicted = predicted.to(dtype=torch.float32)

                all_preds.extend(predicted.detach().cpu().numpy().astype(int).tolist())
                all_scores.extend(scores.detach().cpu().numpy().tolist())

                correct += (predicted == batch_targets).sum().item()

        avg_loss = running_loss / len(dataloader) if len(dataloader) > 0 else 0.0
        accuracy = correct / total if total > 0 else 0.0
        result = {
            "loss": float(avg_loss),
            "accuracy": float(accuracy),
            "sample_ids": all_sample_ids,
            "targets": all_targets,
            "preds": all_preds,
            "scores": all_scores,
        }

        self._print_result(result)
        self._save_result(result)
        print("Evaluation completed!")
        return result

    def _print_result(self, result: dict):
        print("Validation result details:")
        print(f"accuracy: {result['accuracy']:.4f}")

        print("=" * 80)
        print("\nAccuracy result:")
        print("=" * 80 + "\n")
        print(f"{result['accuracy']:.4f}")

    def _save_result(self, result: dict):
        save_dir = os.path.join(self.args.assets_path, "example")
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"validation_result.json")
        Metric.save_metrics(
            result,
            save_dir,
            f"validation_result.json",
            epoch=0,
            c_time='1',
            mode='w'
        )
        print(f"Validation result saved to: {save_path}")

In [4]:
# ====================== 验证参数配置 ======================
# 请根据你的实际路径和训练时的参数修改以下配置
args = Namespace(
    # 数据相关参数（必须与训练时一致）
    dataset_path="../example_data/t1/", 
    seq_len=72,                       
    reduction="mean",                  
    
    # 模型相关参数
    pretrained_model_path="pretrained/Task1/best_model.pth",  
    lora=True,                        
    
    # 验证相关参数
    batch_size=256,                    
    seed=77,                          
    
    assets_path="../example_data/t1/result/",    
)

# ====================== 执行验证 ======================
if __name__ == "__main__":
    control_randomness(args.seed)
    validator = Circadian_Validator(args)
    validation_result = validator.evaluate()

Model configuration initialized
Base CIRCAFM model initialized
Read PeftModel parameters, total keys: 160
PeftModel format verified, sample key: base_model.model.data_embedding.mask_embedding...
Added LoRA structure to base model, converted to PeftModel
Pretrained weights loaded strictly successfully!
  No missing keys
  No unexpected keys
Trainable param stats: mean=0.000054, std=0.106171
Param 6 first 5 values: [ 0.06358685 -0.03613254  0.03058839  0.02279649  0.14248356]
Model loaded and set to evaluation mode
Loading validation dataset: ../example_data/t1/
Current working directory: d:\github\CircaFM\CircaFM_code
Found 1 dataset directories containing .ts files
['../example_data/t1\\']
Validation dataset loaded: 1000 samples, 4 batches
Loss function initialized
Starting evaluation...


100%|██████████| 4/4 [00:01<00:00,  3.88it/s]

Validation result details:
accuracy: 0.9620

Accuracy result:

0.9620
Epoch 0 results saved to ../example_data/t1/result/example\1\validation_result.json
Validation result saved to: ../example_data/t1/result/example\validation_result.json
Evaluation completed!


In [5]:
import utils.pro_json as pro_json

acc, pre, rec, f1, auroc, aupr = pro_json.calculate_binary_metrics(json_path='../example_data/t1/result/example/1/validation_result.json')
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {pre:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUROC: {auroc:.4f}")
print(f"AUPR: {aupr:.4f}")

Accuracy: 0.9620
Precision: 0.9602
Recall: 0.9640
F1 Score: 0.9621
AUROC: 0.9928
AUPR: 0.9920
